In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd 
from shapely import wkt
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

In [3]:
#gdf = pd.DataFrame(gpd.read_file("recintos.gpkg"))
votos = pd.DataFrame(pd.read_excel("datos/votos.xlsx"))
df = (votos.
           pipe(lambda x: x[x["NombrePais"]=="BOLIVIA"]).
           pipe(lambda x: x[x["Descripcion"]=="PRESIDENTE"]).
           #merge(gdf,how="left",left_on=["CodigoLocalidad","CodigoRecinto"], right_on = ["idloc","recinto_codigo"]).
           rename(columns= {"Voto10":"v_PDC","Voto5":"v_LIBRE","Voto9":"v_UNIDAD","Voto1":"v_POPULAR",
                            "Voto3":"v_SUMATE","Voto7":"v_MAS","Voto6":"v_UCS","Voto2":"v_ADN"}).
           assign( c_ut = lambda x: x['CodigoDepartamento'].apply(lambda x: f"{x:02d}") +
                                    x['CodigoProvincia'].apply(lambda x: f"{x:02d}")+
                                    x['CodigoSeccion'].apply(lambda x: f"{x:02d}"))
           )


df

,CodigoMesa,Descripcion,CodigoPais,NombrePais,CodigoDepartamento,NombreDepartamento,CodigoCircunscripcionU,CodigoCircunscripcionE,CodigoProvincia,NombreProvincia,...,Voto12,VotoValido,VotoBlanco,VotoNuloDirecto,VotoNuloDeclinacion,TotalVotoNulo,VotoEmitido,VotoValidoReal,VotoEmitidoReal,c_ut
1227,1000011,PRESIDENTE,32,BOLIVIA,1,Chuquisaca,3,0,1,Oropeza,...,0,30,11,55,0,55,96,30,96,010101
1228,1000021,PRESIDENTE,32,BOLIVIA,1,Chuquisaca,3,0,1,Oropeza,...,0,95,13,90,0,90,198,95,198,010101
1229,1000031,PRESIDENTE,32,BOLIVIA,1,Chuquisaca,3,0,1,Oropeza,...,0,15,1,13,0,13,29,15,29,010101
1230,1000041,PRESIDENTE,32,BOLIVIA,1,Chuquisaca,3,0,1,Oropeza,...,0,102,17,82,0,82,201,102,201,010101
1231,1000051,PRESIDENTE,32,BOLIVIA,1,Chuquisaca,3,0,1,Oropeza,...,0,114,19,70,0,70,203,114,203,010101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35248,9003901,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,0,100,64,26,0,26,190,100,190,090503
35249,9003911,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,0,121,7,9,0,9,137,121,137,090503
35250,9003921,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,0,136,6,39,1,40,182,136,182,090503
35251,9003931,PRESIDENTE,32,BOLIVIA,9,Pando,63,0,5,Federico Román,...,0,21,5,15,1,16,42,21,42,090503


In [28]:
df.groupby(["NombreDepartamento","CodigoDepartamento"]).agg({"CodigoMesa":"sum"})

,,CodigoMesa
NombreDepartamento,CodigoDepartamento,
Beni,8,10993433988
Chuquisaca,1,1831482015
Cochabamba,3,19239396676
La Paz,2,18612013723
Oruro,4,6786341411
Pando,9,3546778577
Potosí,5,11913264907
Santa Cruz,7,64220471064
Tarija,6,10900463908


In [30]:
df.filter(like = "v_").sum().apply(lambda x: "{:,}".format(x))

v_Andronico        456,002
v_Pavel             77,576
v_Manfred          361,640
v_LIBRE          1,430,176
v_JhonnyFer         89,253
v_DelCastillo      169,887
v_UNIDAD         1,054,568
v_PDC            1,717,432
dtype: object

In [23]:
df["NombreMunicipio"]

0                                           Sucre
1                                           Sucre
2                                           Sucre
3                                           Sucre
4                                           Sucre
                           ...                   
34021    Santos Mercado                          
34022    Santos Mercado                          
34023    Santos Mercado                          
34024    Santos Mercado                          
34025    Santos Mercado                          
Name: NombreMunicipio, Length: 34026, dtype: object

In [13]:
import pandas as pd
import geopandas as gpd


voto_cols = [col for col in df.columns if col.startswith("v_")]


resumen = (
    df.
    groupby(["NombreMunicipio","c_ut"])[voto_cols].sum().reset_index().
    assign( ganador = lambda x:x[voto_cols].idxmax(axis=1) ).
    assign( Total = lambda x:x.loc[:, 'v_POPULAR':'v_PDC'].sum(1)).
    assign(**{col + '_porcentaje' : lambda x, c=col: x[c] / x['Total'] * 100 for col in voto_cols})
    
    )

gdf = gpd.read_file("shapes/municipios/municipios339.shp", encoding='latin1')  
import re

def fix_encoding(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].apply(
            lambda x: x.encode('latin1').decode('utf-8').strip() if isinstance(x, str) else x
        )
    return df

gdf = fix_encoding(gdf)

cambios = {
    'AIOC Charagua Iyambae': 'Charagua',
    'AIOC Uru Chipaya' : 'Chipaya',
    'AIOC de Salinas' : 'Salinas de Garcí Mendoza',
    'Icla' : 'Villa Ricardo Mugia - Icla',
    'Pampagrande' : 'Pampa Grande',
    'Santa Cruz de La Sierra':'Santa Cruz de la Sierra',
    'Santiago de Huayllamarca' : 'Huayllamarca',
    'Sopachuy':'Sopachui',
    'TIOC Guaraní Chaqueño de Huacaya' : 'Huacaya',
    'AIOC Guaraní Kereimba Iyaambae' : 'Gutiérrez'
}


resumen["NombreMunicipio"] = resumen["NombreMunicipio"].apply(lambda x: x.strip()).replace(cambios)




gdf = gdf.rename(columns={"NOM_MUN": "NombreMunicipio"}) 

merged = gdf.merge(resumen, on = "c_ut", how="outer")
merged["NombreMunicipio"] = merged['NombreMunicipio'].fillna(merged["c_ut"])
merged = merged.drop(columns=["ogc_fid","id","objectid","shape_leng","shape_area"])
merged.to_file("municipios_votacion.geojson", driver="GeoJSON")

In [16]:
merged.head()

,c_ut,departamen,provincia,municipio,capital,geometry,NombreMunicipio,v_POPULAR,v_ADN,v_SUMATE,...,ganador,Total,v_POPULAR_porcentaje,v_ADN_porcentaje,v_SUMATE_porcentaje,v_LIBRE_porcentaje,v_UCS_porcentaje,v_MAS_porcentaje,v_UNIDAD_porcentaje,v_PDC_porcentaje
0,010101,Chuquisaca,Oropeza,Sucre,Sucre,"POLYGON ((-65.14702 -18.61252, -65.14677 -18.6...",Sucre,8383.0,2193.0,9779.0,...,v_LIBRE,180602.0,4.641698,1.214272,5.414669,37.121405,0.750269,3.101295,13.652119,34.104273
1,010102,Chuquisaca,Oropeza,Yotala,Yotala,"POLYGON ((-65.24691 -19.09556, -65.24712 -19.0...",Yotala,466.0,46.0,109.0,...,v_PDC,3086.0,15.100454,1.490603,3.532080,17.952041,2.203500,4.633830,8.263124,46.824368
2,010103,Chuquisaca,Oropeza,Poroma,Poroma,"POLYGON ((-65.65823 -18.36605, -65.65684 -18.3...",Poroma,816.0,63.0,107.0,...,v_POPULAR,2058.0,39.650146,3.061224,5.199223,4.616132,3.547133,4.907677,2.769679,36.248785
3,010201,Chuquisaca,Azurduy,Azurduy,Villa Azurduy,"POLYGON ((-64.36172 -19.8297, -64.34352 -19.83...",Azurduy,236.0,54.0,135.0,...,v_PDC,1835.0,12.861035,2.942779,7.356948,18.474114,2.888283,15.749319,5.613079,34.114441
4,010202,Chuquisaca,Azurduy,Tarvita,Villa Orías,"POLYGON ((-64.53151 -19.55003, -64.52247 -19.5...",Tarvita,477.0,70.0,123.0,...,v_PDC,2304.0,20.703125,3.038194,5.338542,7.725694,2.039931,4.557292,3.645833,52.951389


In [36]:
a=merged.pipe(lambda x: x[x["NombreMunicipio"].isna() | x["municipio"].isna()])
a

,ogc_fid,id,objectid,c_ut,departamen,provincia,municipio,capital,shape_leng,shape_area,...,ganador,Total,v_POPULAR_porcentaje,v_ADN_porcentaje,v_SUMATE_porcentaje,v_LIBRE_porcentaje,v_UCS_porcentaje,v_MAS_porcentaje,v_UNIDAD_porcentaje,v_PDC_porcentaje
155,NaN,NaN,NaN,031304,NaN,NaN,NaN,NaN,NaN,NaN,...,v_POPULAR,565.0,69.557522,2.123894,6.371681,1.769912,4.070796,2.654867,0.884956,12.566372
214,NaN,NaN,NaN,050405,NaN,NaN,NaN,NaN,NaN,NaN,...,v_PDC,2777.0,32.481095,3.709039,5.545553,4.933381,5.437523,5.941664,2.988837,38.962910
234,NaN,NaN,NaN,051204,NaN,NaN,NaN,NaN,NaN,NaN,...,v_PDC,1746.0,17.812142,1.775487,4.238259,9.049255,5.326460,4.410080,6.300115,51.088202
319,NaN,NaN,NaN,080502,NaN,NaN,NaN,NaN,NaN,NaN,...,v_POPULAR,622.0,42.443730,1.768489,6.109325,5.305466,4.340836,9.163987,13.183280,17.684887
343,6.0,57.0,774.0,Lago,None,None,None,None,7.107091e+04,1.140906e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
344,118.0,59.0,776.0,Lago,None,None,None,None,2.514942e+05,1.276659e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
345,237.0,176.0,2083.0,Lago,None,None,None,None,8.649334e+05,3.099824e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
346,50.0,34.0,753.0,Salar,None,None,None,None,1.033956e+06,9.349140e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
347,54.0,38.0,757.0,Salar,None,None,None,None,4.210488e+05,2.125332e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
#simulacion segunda vuelta
import numpy as np

np.random.seed(1234) 

merged_2 = (merged.
           drop(columns=[col for col in merged.columns if col.startswith("v_") or col == "ganador"]).
           assign(v_LIBRE = lambda x: round(np.random.uniform(0, 1) * x["Total"],0),
                  v_PDC = lambda x: x["Total"] - x["v_LIBRE"])
)

voto_cols = [col for col in merged_2.columns if col.startswith("v_")]

merged_2 = (
    merged_2.
    assign(**{col + '_porcentaje' : lambda x, c=col: x[c] / x['Total'] * 100 for col in voto_cols})
)
merged_2

merged.to_file("municipios_votacion_segunda_vuelta.geojson", driver="GeoJSON")